In [ ]:
import mteb
from sentence_transformers import SentenceTransformer

# Select model
model_name = "sentence-transformers/all-MiniLM-L6-v2"
# model_name = "intfloat/multilingual-e5-large"
model = mteb.get_model(model_name) # if the model is not implemented in MTEB it will be eq. to SentenceTransformer(model_name)

# local_models = {
#     'all-mpnet-base-v2': SentenceTransformer('all-mpnet-base-v2'),
#     'all-MiniLM-L6-v2': SentenceTransformer('all-MiniLM-L6-v2'),
#     'e5-base': SentenceTransformer('intfloat/e5-base-v2')
# }

# Select tasks
# tasks = mteb.get_tasks(tasks=["Banking77Classification.v2"])
tasks = mteb.get_tasks(tasks=[
    # 'STS12', 
    # 'STS13', 
    # # 'STS14',
    # # 'STS15',
    # # 'STS16',
    # 'STSBenchmark', 
    'SICK-R'
    ])

# evaluate
results = mteb.evaluate(model, tasks=tasks)

Evaluating task SICK-R: 100%|██████████| 4/4 [00:00<00:00, 666.85it/s]


In [49]:
df = results.to_dataframe(format="long")
df

c:\Users\Utilisateur\Documents\Veille_Technique\env_veille_tech\Lib\site-packages\mteb\results\model_result.py:68: FutureWarning: The provided callable <function mean at 0x0000027CA1023C40> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  .agg(score=("score", aggregation_fn))


,model_name,task_name,score
0,sentence-transformers/all-MiniLM-L6-v2,SICK-R,0.775824
1,sentence-transformers/all-MiniLM-L6-v2,STS12,0.723690
2,sentence-transformers/all-MiniLM-L6-v2,STS13,0.806032
3,sentence-transformers/all-MiniLM-L6-v2,STSBenchmark,0.820325


In [54]:
results.task_results[0].scores

{'test': [{'pearson': 0.8136,
   'spearman': 0.723694,
   'cosine_pearson': 0.8136,
   'cosine_spearman': 0.72369,
   'manhattan_pearson': 0.774827,
   'manhattan_spearman': 0.723646,
   'euclidean_pearson': 0.774443,
   'euclidean_spearman': 0.723672,
   'main_score': 0.72369,
   'hf_subset': 'default',
   'languages': ['eng-Latn']}]}

In [ ]:
import mteb
from sentence_transformers import SentenceTransformer
from typing import List

# Liste des modèles à évaluer
model_names = [
    "sentence-transformers/all-MiniLM-L6-v2",
    "intfloat/multilingual-e5-large"
]

# Tâches d'évaluation
tasks = mteb.get_tasks(tasks=[
    'STSBenchmark', 
    'SICK-R'
])

# Stockage des résultats
all_results = {}

# Boucle d'évaluation
for model_name in model_names:
    print(f"\n{'='*60}")
    print(f"Évaluation du modèle : {model_name}")
    print('='*60)
    
    try:
        # Chargement du modèle via SentenceTransformer (recommandé pour plus de contrôle)
        model = SentenceTransformer(model_name)
        
        # Évaluation
        evaluation = mteb.MTEB(tasks=tasks)
        results = evaluation.run(
            model,
            output_folder=f"results/{model_name.replace('/', '_')}",
            overwrite_results=True
        )
        
        # Extraction des scores principaux pour affichage comparatif
        scores = {}
        for task_result in results:
            task_name = task_result["mteb_dataset_name"]
            # Pour STS/SICK-R, le score principal est souvent sous 'spearman' ou 'cos_sim'
            if "scores" in task_result:
                main_score = task_result["scores"]["test"][0].get("spearman", 
                                 task_result["scores"]["test"][0].get("cos_sim", None))
                if main_score is not None:
                    scores[task_name] = round(main_score * 100, 2)  # Conversion en %
        
        all_results[model_name] = scores
        print(f"✓ Évaluation terminée pour {model_name}")
        print(f"Scores : {scores}")
        
    except Exception as e:
        print(f"✗ Erreur lors de l'évaluation de {model_name} : {e}")
        all_results[model_name] = {"error": str(e)}

# Affichage comparatif final
print("\n" + "="*60)
print("RÉSULTATS COMPARATIFS")
print("="*60)

# En-têtes
task_list = sorted(set(task for scores in all_results.values() if isinstance(scores, dict) for task in scores))
header = f"{'Modèle':<45} " + " ".join([f"{task:<20}" for task in task_list])
print(header)
print("-" * (45 + 22 * len(task_list)))

# Lignes de résultats
for model_name, scores in all_results.items():
    if isinstance(scores, dict) and "error" not in scores:
        row = f"{model_name:<45} "
        for task in task_list:
            score = scores.get(task, "N/A")
            row += f"{str(score) + ' %':<20}"
        print(row)
    else:
        print(f"{model_name:<45} Erreur : {scores.get('error', 'Inconnue')}")

print("="*60)

In [36]:
# Stockage des résultats
all_results = {}

# Boucle d'évaluation
for model_name in model_names:
    print(f"\n{'='*60}")
    print(f"Évaluation du modèle : {model_name}")
    print('='*60)
    
    try:
        # Chargement du modèle via SentenceTransformer (recommandé pour plus de contrôle)
        model = SentenceTransformer(model_name)
        
        # Évaluation
        evaluation = mteb.MTEB(tasks=tasks)
        results = evaluation.run(
            model,
            output_folder=f"results/{model_name.replace('/', '_')}",
            overwrite_results=True
        )
        
        # Extraction des scores principaux pour affichage comparatif
        scores = {}
        for task_result in results:
            print(type(task_result))
            print(task_result)

    except Exception as e:
        print(f"✗ Erreur lors de l'évaluation de {model_name}")


Évaluation du modèle : sentence-transformers/all-MiniLM-L6-v2


C:\Users\Utilisateur\AppData\Local\Temp\ipykernel_13336\4131451556.py:15: DeprecationWarning: MTEB is deprecated and will be removed in future versions. Please use the `mteb.evaluate` function instead.
  evaluation = mteb.MTEB(tasks=tasks)


───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

STS

- STSBenchmark, t2t

...

───────────────────────────────────────────────── Selected tasks  ─────────────────────────────────────────────────

STS

- STSBenchmark, t2t

- SICK-R, t2t

<class 'mteb.results.task_result.TaskResult'>
dataset_revision='b0fddb56ed78048fa8b90373c8a3cfc37b684831' task_name='STSBenchmark' mteb_version='2.7.2' scores={'test': [{'pearson': 0.8359201417733226, 'spearman': 0.8547779409975763, 'cosine_pearson': 0.8359201600437254, 'cosine_spearman': 0.8547778858529195, 'manhattan_pearson': 0.8457158724223774, 'manhattan_spearman': 0.853054807389951, 'euclidean_pearson': 0.8470932230832279, 'euclidean_spearman': 0.8547779409975763, 'main_score': 0.8547778858529195, 'hf_subset': 'default', 'languages': ['eng-Latn']}]} evaluation_time=519.9367451667786 kg_co2_emissions=None
<class 'mteb.results.task_result.TaskResult'>
dataset_revision='20a6d6f312dd54037fe07a32d58e5e168867909d' task_name='SICK-R' mteb_version='2.7.2' scores={'test': [{'pearson': 0.8375964859989327, 'spearman': 0.8054975475437268, 'cosine_pearson': 0.837596481395501, 'cosine_spearman': 0.8054973543372017, 'manhattan_pearson': 0.8049264897844923, 'manhattan_spearman': 0.80411961981790